# Task 4 — Random Forests & Ensembles: bagging, voting, stacking and AdaBoost on a spam filter

MOP 3231 · Machine Learning (Python) · Abror Imamsidikov

Data: UCI Spambase (Hopkins, Reeber, Forman & Suermondt, 1999; CC BY 4.0; doi:10.24432/C53G6X), deduplicated by the
instructor: 4204 e-mails, 57 numeric features, label `spam` (1 = spam, 0 = legitimate mail). Spam is the positive class;
a false positive is a legitimate mail flagged as spam.

## Part 0 — Setup

In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.12.12 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 2954          # last 4 digits of your student ID, as an integer (0421 -> 421)
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


**SEED = 2954** (the last four digits of my student ID). It seeds the train/test split, the folds `cv` and every
random model below. The check line prints `(4204, 57) 39.9 (3153, 57) (1051, 57)` as the handout requires.

### The fixed models (exactly as given in the handout)

In [3]:
import math

import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]   # members of voting and stacking in Parts 3-4

In [4]:
from sklearn.model_selection import cross_validate

n_train, n_test = len(y_train), len(y_test)


def cv_scores(model):
    """Accuracy on each of the five folds of cv on the training split."""
    return cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)["test_score"]


def fold_table(scores):
    """Table: one row per model, the five fold scores, mean and std (ddof=0)."""
    t = pd.DataFrame(scores).T
    t.columns = [f"fold {i + 1}" for i in range(t.shape[1])]
    t["mean"] = t.iloc[:, :5].mean(axis=1)
    t["std"] = t.iloc[:, :5].std(axis=1, ddof=0)
    return t


def paired(a, b):
    """Paired differences a - b (per fold or per draw) with mean, std and the 'inside the spread' verdict."""
    d = np.asarray(a) - np.asarray(b)
    row = {f"#{i + 1}": v for i, v in enumerate(d)}
    row["mean"], row["std"] = d.mean(), d.std()
    row["inside spread"] = bool(abs(d.mean()) <= d.std())
    return row

`cv_scores` is the handout's `cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]`
(with `n_jobs=-1` only for speed; it clones the model and uses the same folds). `paired` returns the paired
differences, their mean and std (`np.std`, ddof=0), and whether the difference is **inside the fold spread**
(|mean| ≤ std), in which case no winner may be named.

## Part 1 — Bagging and out-of-bag estimates

In [5]:
def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices.

    sample_indices: list or 1-D integer array of row numbers in [0, n_samples - 1], repeats allowed, may be empty.
    Returns a new bool array of shape (n_samples,); element i is True exactly when i is not in sample_indices.
    """
    idx = np.asarray(sample_indices, dtype=np.int64).reshape(-1)   # a new array; the argument is not touched
    mask = np.ones(int(n_samples), dtype=bool)
    mask[idx] = False
    return mask

In [6]:
# the examples of the handout are part of the rules
assert oob_mask([0, 2, 2], 5).tolist() == [False, True, False, True, True]
assert oob_mask([], 3).tolist() == [True, True, True]
assert oob_mask(np.array([3, 0]), 4).tolist() == [False, True, True, False]
_a = np.array([3, 0]); oob_mask(_a, 4); assert _a.tolist() == [3, 0]          # argument unchanged
assert oob_mask([1], 2).dtype == bool and oob_mask([1], 2).shape == (2,)
print("oob_mask: all examples pass")

oob_mask: all examples pass


### 1.1 Out-of-bag share

In [7]:
bag.fit(X_train, y_train)
n = n_train
oob_share = np.array([oob_mask(s, n).mean() for s in bag.estimators_samples_])
print(f"n = {n} training rows, {len(bag.estimators_)} trees")
print(f"mean out-of-bag share per tree : {oob_share.mean():.4f}   formula (1 - 1/n)^n     = {(1 - 1 / n) ** n:.4f}")
print(f"mean in-bag share per tree     : {1 - oob_share.mean():.4f}   formula 1 - (1 - 1/n)^n = {1 - (1 - 1 / n) ** n:.4f}")
print(f"(range over the 100 trees: {oob_share.min():.4f} to {oob_share.max():.4f})")

n = 3153 training rows, 100 trees
mean out-of-bag share per tree : 0.3685   formula (1 - 1/n)^n     = 0.3678
mean in-bag share per tree     : 0.6315   formula 1 - (1 - 1/n)^n = 0.6322
(range over the 100 trees: 0.3543 to 0.3825)


### 1.2 Out-of-bag accuracy by hand

In [8]:
votes = np.zeros((n, 2))
for est, sample in zip(bag.estimators_, bag.estimators_samples_):
    m = oob_mask(sample, n)
    pred = est.predict(X_train[m]).astype(int)
    np.add.at(votes, (np.flatnonzero(m), pred), 1)       # one vote per tree for which the row is out of bag
oob_pred = np.argmax(votes, axis=1)                         # tie -> class 0
my_oob_acc = np.mean(oob_pred == y_train)
print(f"rows with no out-of-bag vote: {(votes.sum(axis=1) == 0).sum()}")
print(f"my out-of-bag accuracy = {my_oob_acc:.6f}")
print(f"bag.oob_score_         = {bag.oob_score_:.6f}")
print("equal:", my_oob_acc == bag.oob_score_)

rows with no out-of-bag vote: 0
my out-of-bag accuracy = 0.941326
bag.oob_score_         = 0.941326
equal: True
